# 🫀 PULSE ECG — TCN-BiLSTM-Attention Training (GitHub + Direct Download)
**Runtime → Change runtime type → GPU (T4)** before running!

This notebook:
1. Clones your code from GitHub directly (no Google Drive needed)
2. Downloads raw ECG datasets directly from PhysioNet
3. Processes, augments, and trains the model
4. Lets you download the trained model as a zip

---
## STEP 1: Clone Project from GitHub

In [ ]:
!git clone https://github.com/Gowtham-1301/TCN_BiLSTM.git /content/project
%cd /content/project

import os
for d in ['data/raw', 'data/processed', 'models/tfjs_model', 'logs', 'evaluation', 'checkpoints']:
    os.makedirs(d, exist_ok=True)

print('\n✅ Project code cloned successfully!')
print('📁 Files:', [f for f in os.listdir('.') if not f.startswith('.')])

---
## STEP 2: Install Dependencies

In [ ]:
!pip install -q wfdb scipy scikit-learn matplotlib seaborn tqdm PyYAML h5py pandas tensorboard
print('\n✅ All dependencies installed')

---
## STEP 3: Verify GPU

In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
print(f'TensorFlow: {tf.__version__}')
print(f'GPU: {gpus}')

if gpus:
    !nvidia-smi
    print('\n✅ GPU is ready!')
else:
    print('\n❌ No GPU detected!')
    print('Go to: Runtime → Change runtime type → GPU (T4)')

---
## STEP 4: Download Raw ECG Datasets from PhysioNet
**FAST PROTOTYPING MODE:** We only download the MIT-BIH dataset (~230 MB, 1 min) to train instantly.
*To train on all 14GB of data later, simply uncomment the other datasets in the code below.*

In [ ]:
%%time
import os

datasets = {
    'mitdb': {
        'url': 'https://physionet.org/files/mitdb/1.0.0/',
        'dest': 'data/raw/mitdb',
        'name': 'MIT-BIH Arrhythmia Database',
        'size': '~230 MB'
    },
    # Uncomment the below datasets for a full, production-ready training run (~15 mins download)
    # 'ludb': {
    #     'url': 'https://physionet.org/files/ludb/1.0.1/',
    #     'dest': 'data/raw/ludb',
    #     'name': 'Lobachevsky University DB',
    #     'size': '~3.2 GB'
    # },
    # 'cpsc2018': {
    #     'url': 'https://physionet.org/files/cpsc2018/1.0.0/',
    #     'dest': 'data/raw/CPSC-2018',
    #     'name': 'CPSC 2018',
    #     'size': '~2.1 GB'
    # },
    # 'ptbxl': {
    #     'url': 'https://physionet.org/files/ptb-xl/1.0.3/',
    #     'dest': 'data/raw/PTB-XL',
    #     'name': 'PTB-XL',
    #     'size': '~8.5 GB'
    # },
}

for key, ds in datasets.items():
    dest = ds['dest']
    os.makedirs(dest, exist_ok=True)

    existing_files = sum(1 for _ in os.walk(dest) for f in _[2])
    if existing_files > 10:
        print(f'\n✅ {ds["name"]} already downloaded ({existing_files} files) — skipping')
        continue

    print(f'\n{"="*60}')
    print(f'📥 Downloading {ds["name"]} ({ds["size"]})...')
    print(f'{"="*60}')

    !wget -r -N -c -np -nH --cut-dirs=3 --no-check-certificate \
      -q --show-progress \
      '{ds["url"]}' \
      -P '{dest}'

    n_files = sum(1 for _ in os.walk(dest) for f in _[2])
    print(f'✅ {ds["name"]}: {n_files} files downloaded')

print(f'\n{"="*60}')
print('🎉 ALL DATASETS DOWNLOADED!')
print(f'{"="*60}')

---
## STEP 5: Train the Model 🚀
Runs preprocessing → beat extraction → augmentation → training → evaluation.
**Expected time: ~30–45 min on T4 GPU.**

In [ ]:
!python train.py --skip-download

---
## STEP 6: View Training Results

In [ ]:
import json, glob, os
from IPython.display import Image, display

for ef in sorted(glob.glob('evaluation/*.json')):
    print(f'\n{"="*50}')
    print(f'📊 {os.path.basename(ef)}')
    with open(ef) as f:
        data = json.load(f)
        for k, v in data.items():
            if isinstance(v, float):
                print(f'  {k}: {v:.4f}')

for pf in sorted(glob.glob('evaluation/*.png')):
    print(f'\n📈 {os.path.basename(pf)}')
    display(Image(filename=pf, width=600))

---
## STEP 7: Convert to TensorFlow.js

In [ ]:
!pip install -q tensorflowjs
!python convert_to_tfjs.py --metadata

tfjs_dir = 'models/tfjs_model'
if os.path.exists(tfjs_dir):
    print(f'\n✅ TF.js model files:')
    for f in os.listdir(tfjs_dir):
        size = os.path.getsize(os.path.join(tfjs_dir, f)) / 1024
        print(f'  {f}: {size:.1f} KB')
else:
    print('❌ TF.js conversion may have failed')

---
## STEP 8: Download Trained Models & Logs

In [ ]:
import shutil
from google.colab import files

# Create downloadable zip of models, evaluation, and logs
shutil.make_archive('/content/pulse_trained_output', 'zip', '/content/project', 'models')

size = os.path.getsize('/content/pulse_trained_output.zip') / (1024*1024)
print(f'📦 Output zip created: {size:.1f} MB')

print('Downloading to your computer...')
files.download('/content/pulse_trained_output.zip')